# Rebuilding the labels and the frames from the audio (level 3)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/huseyin-karaca/hit-asr/blob/main/notebooks/extract.ipynb)

The routers learn from two things per corpus and expert: the expert's **labels** (its transcripts, their word-error
counters against the reference, and the mean of its encoder states) and its **frames** (the frame-level encoder states
themselves, fp16, at the encoder's own frame rate). Levels 1 and 2 read both from the published dataset; this
notebook rebuilds them from the audio: every expert of every corpus's trio decodes the corpus in full and its encoder
states are stored.

**Where it writes.** A Hugging Face dataset repository of your own; the published one is read-only. This is the only
notebook that needs a Hugging Face token, because it writes to that repository and because Cohere Transcribe is gated
on the Hub (accept its terms there first, with the account whose token you use). Before running:

1. create an empty dataset repository on the Hub, say `<you>/hit-asr-rebuild`;
2. add a write token as the Colab secret `HF_TOKEN` (or the environment variable of that name);
3. set `REBUILD_REPO` below to that repository.

Then run `main_<corpus>` at level 3 with `HITASR_HUB=<you>/hit-asr-rebuild` set before its imports: it reads the
labels and frames from your repository and the published searches from the public one, and retrains every fold.

**Cost.** Each (corpus, expert) is one pass of the model over the corpus: minutes to an hour on a 96 GB GPU, with
1-13 GB of frames each. The audio is read from the published `<corpus>_base` configurations (16 kHz, the clips the
paper uses). An expert whose labels and frames are already complete in your repository is skipped.

In [ ]:
import importlib.util, subprocess, sys
if importlib.util.find_spec("hitasr") is None:     # Colab: install the package (a local or Docker run already has it)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "hit-asr[paper] @ git+https://github.com/huseyin-karaca/hit-asr.git@main"], check=True)

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "datasets<4.0", "jiwer", "librosa", "soundfile",
                "transformers>=4.57", "accelerate", "sentencepiece", "torchaudio", "mistral-common[audio]", "peft"],
               check=True)

In [ ]:
REBUILD_REPO = ""      # "<you>/hit-asr-rebuild": an empty dataset repository of yours

import os
if not REBUILD_REPO:
    raise RuntimeError("set REBUILD_REPO to a dataset repository of yours (the published one is read-only)")
os.environ["HITASR_HUB"] = os.environ["HITASR_RECORDS"] = REBUILD_REPO    # before hitasr is imported

from hitasr.configs import REBUILD
from hitasr.core import REPO_ID
from hitasr.extractor import compare_with_published, push_leftovers, rebuild_plan, sweep
from hitasr.models.registry import load_expert_classes
from labkit.env import describe_env, login_hf

assert REPO_ID == REBUILD_REPO, "hitasr was imported before REBUILD_REPO was set: restart the runtime"
login_hf()
describe_env()

## 1. What to rebuild

Every corpus of the paper and its trio (`hitasr.configs.REBUILD`).

In [ ]:
plan = rebuild_plan(REBUILD.corpora)

## 2. The pass

Per corpus the audio is loaded once; then per expert a short smoke test, the batch fit (the longest clips are pushed
through once and the batch is halved until they fit), the pass, and the upload of the labels and the frame shards.
Every clip is decoded; nothing is adopted from an existing label set.

In [ ]:
log = sweep(plan, load_expert_classes(), REBUILD.batch, min_batch=REBUILD.min_batch, decode="full")
push_leftovers(wait_minutes=75)
log

## 3. Check against the published pass

The same checkpoint, decoder and normaliser give the same transcripts up to the GPU's numerics, so each expert's
corpus WER should match the published extraction record to about the third decimal.

In [ ]:
compare_with_published(log)